## JSON Parsing and Processing

In [1]:
import json
import os
os.makedirs("data/json_files", exist_ok = True)

In [2]:
# Sample nested JSON data
json_data = {
  "company": "TechCorp",
  "employees": [
    {
      "id": 1,
      "name": "John Doe",
      "role": "Software Engineer",
      "skills": [
        "Python",
        "JavaScript",
        "React"
      ],
      "projects": [
        {
          "name": "RAG System",
          "status": "In Progress"
        },
        {
          "name": "Data Pipeline",
          "status": "Completed"
        }
      ]
    },
    {
      "id": 2,
      "name": "Jane Smith",
      "role": "Data Scientist",
      "skills": [
        "Python",
        "Machine Learning",
        "SQL"
      ],
      "projects": [
        {
          "name": "ML Model",
          "status": "In Progress"
        },
        {
          "name": "Analytics Dashboard",
          "status": "Planning"
        }
      ]
    }
  ],
  "departments": {
    "engineering": {
      "head": "Mike Johnson",
      "budget": 1000000,
      "team_size": 25
    },
    "data_science": {
      "head": "Sarah Williams",
      "budget": 750000,
      "team_size": 15
    }
  }
}

In [3]:
with open("data/json_files/company_data.json", "w") as f:
    json.dump(json_data, f, indent = 2)

In [4]:
jsonl_data = [
    {"timestamp": "2024-01-01", "event": "user_login", "user_id": 123},
    {"timestamp": "2024-01-01", "event": "page_view", "user_id": 123, "page": "/home"},
    {"timestamp": "2024-01-01", "event": "purchase", "user_id": 123, "amount": 99.99}
]

with open("data/json_files/events.jsonl", "w") as f:
    for entry in jsonl_data:
        f.write(json.dumps(entry) + "\n")

## JSON Processing Stratergies

In [5]:
from langchain_community.document_loaders import JSONLoader
import json

# Method 1: JSONLoader with jq_schema
employee_loader = JSONLoader(
    file_path = "data/json_files/company_data.json",
    jq_schema = ".employees[]",
    text_content = False
)

employee_docs = employee_loader.load()
print(f"Loaded {len(employee_docs)} employee documents")
print(f"First employee: {employee_docs[0].page_content[:200]}...")


C:\Users\CHITTA\AppData\Local\Temp\ipykernel_19112\914722916.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader
c:\Users\CHITTA\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 2 employee documents
First employee: {"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status"...


In [7]:
# Method 2: Custom JSON preocessing for complex structure
from typing import List
from langchain_core.documents import Document

print("\nCustom JSON Processing")

def process_json_intelligently(filepath: str) -> List[Document]:
    """Process JSON with intelligence flattening and context preservation"""
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    documents = []
    
    # Example: Process nested JSON structure
    for item in data.get('employees', []):
        content = f"""Employee Profile: 
        Name: {item['name']}
        Role: {item['role']}
        Skills: {', '.join(item['skills'])}
        
        Projects: """
        for proj in item.get('projects', []):
            content += f"- {proj['name']}: (Status: {proj['status']})\n"
            doc = Document(
                page_content = content,
                metadata = {
                    "source": filepath,
                    "data_type": "employee_profile",
                    "employee_id": item['id'],
                    "employee_name": item['name'],
                    "role": item['role']
                }
            )
            documents.append(doc)
    
    return documents


Custom JSON Processing


In [8]:
process_json_intelligently("data/json_files/company_data.json")

[Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 1, 'employee_name': 'John Doe', 'role': 'Software Engineer'}, page_content='Employee Profile: \n        Name: John Doe\n        Role: Software Engineer\n        Skills: Python, JavaScript, React\n\n        Projects: - RAG System: (Status: In Progress)\n'),
 Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 1, 'employee_name': 'John Doe', 'role': 'Software Engineer'}, page_content='Employee Profile: \n        Name: John Doe\n        Role: Software Engineer\n        Skills: Python, JavaScript, React\n\n        Projects: - RAG System: (Status: In Progress)\n- Data Pipeline: (Status: Completed)\n'),
 Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 2, 'employee_name': 'Jane Smith', 'role': 'Data Scientist'}, page_content='Employee Profile: \n        